In [1]:
# ==============================================================================
# Script: MetNormalizer Dataset Preprocessing and Batch Summarization
# Description: Parses MetNormalizer demo data (single batch), aligns metadata to 
#              pi-metaboqc standards (imputing missing Batch info), performs 
#              multi-level sorting (Batch & Inject Order), and ensures data 
#              integrity before generating a batch QC summary.
# ==============================================================================

import os
import pandas as pd
from pprint import pprint
from loguru import logger


def process_metnormalizer_dataset(meta_file: str, intensity_file: str) -> tuple:
    """
    Parses MetNormalizer raw CSV files, cleans intensity matrix,
    aligns metadata to pi-metaboqc standards (adding fallback Batch), 
    and strictly sorts both matrices by Batch and Injection Order.
    """
    logger.info(f"Loading MetNormalizer metadata from: {meta_file}")
    
    try:
        meta_raw = pd.read_csv(meta_file)
    except Exception as e:
        logger.error(f"Failed to read metadata file: {e}")
        return None, None

    logger.info("Standardizing metadata columns...")
    
    # 1. Safely identify and map columns based on MetNormalizer conventions
    col_mapping = {}
    for col in meta_raw.columns:
        c_lower = str(col).lower()
        if c_lower == "batch":
            col_mapping[col] = "Batch"
        elif "order" in c_lower:
            col_mapping[col] = "Inject Order"
        elif c_lower == "class":
            col_mapping[col] = "Sample Type"
        elif "sample" in c_lower and "name" in c_lower:
            col_mapping[col] = "Sample Name"
            
    meta = meta_raw.rename(columns=col_mapping)

    logger.info("Formatting metadata values...")

    # 2. Defensively handle missing 'Batch' column (Common in MetNormalizer data)
    if "Batch" not in meta.columns:
        logger.warning("No 'Batch' column found. Defaulting to 'Batch-1'.")
        meta["Batch"] = "Batch-1"
    else:
        # Standardize existing 'Batch' content to ensure 'Batch-X' format
        clean_batch = (
            meta["Batch"].astype(str).str.replace(r"Batch[-_\s]?", "", regex=True)
        )
        meta["Batch"] = "Batch-" + clean_batch

    # 3. Standardize 'Sample Type' content
    if "Sample Type" in meta.columns:
        meta["Sample Type"] = meta["Sample Type"].replace({
            "Subject": "Actual Sample",
            "QC": "QC",
            "Blank": "Blank",
            "subject": "Actual Sample",
            "qc": "QC"
        })

    # Ensure critical column exists
    if "Sample Name" not in meta.columns:
        logger.error("Could not identify 'Sample Name' column in metadata.")
        return None, None
        
    meta = meta[["Batch", "Sample Type", "Inject Order", "Sample Name"]]
    
    logger.info(f"Loading intensity file from: {intensity_file}")
    
    # 4. Process intensity file
    try:
        int_df = pd.read_csv(intensity_file)
        
        # Drop structural columns to prevent matrix pollution
        cols_to_drop = [c for c in ["mz", "rt"] if c in int_df.columns]
        int_df = int_df.drop(columns=cols_to_drop)
            
    except Exception as e:
        logger.error(f"Failed to process intensity file {intensity_file}: {e}")
        return None, None

    # Set feature name as the standard index
    # Note: MetNormalizer usually names the feature column 'name'
    if "name" in int_df.columns:
        int_df = int_df.set_index("name")
    else:
        # Fallback if the first column is the feature ID but not named 'name'
        int_df = int_df.set_index(int_df.columns[0])
        
    int_df.index.name = "Metabolite"
    
    logger.info("Sorting data by Batch and Inject Order...")

    # 5. Multi-level Sorting Logic
    # Ensure Inject Order is numeric for proper sorting
    meta["Inject Order"] = pd.to_numeric(meta["Inject Order"], errors="coerce")
    
    # Sort metadata by Batch (ASCII) and then Inject Order (Numeric)
    meta = meta.sort_values(
        by=["Batch", "Inject Order"],
        ascending=[True, True],
        ignore_index=True
    )
    
    # Extract the sorted sequence of sample names
    sorted_sample_names = meta["Sample Name"].tolist()
    
    # Check if any samples are missing from the intensity matrix before slicing
    missing_in_int = [s for s in sorted_sample_names if s not in int_df.columns]
    if missing_in_int:
        logger.warning(
            f"Found {len(missing_in_int)} samples in meta missing from "
            f"intensity matrix. Examples: {missing_in_int[:3]}"
        )
        # Keep only samples that actually exist in the intensity matrix
        sorted_sample_names = [s for s in sorted_sample_names if s in int_df.columns]
        # Re-filter meta to match available samples
        meta = meta[meta["Sample Name"].isin(sorted_sample_names)].reset_index(drop=True)

    # Reorder intensity columns to strictly match the sorted metadata rows
    int_df = int_df[sorted_sample_names]
    
    return meta, int_df


def summarize_batch_info(
    meta: pd.DataFrame,
    batch_col: str = "Batch",
    order_col: str = "Inject Order",
    type_col: str = "Sample Type"
) -> pd.DataFrame:
    """
    Summarizes batch details including injection order range, sample counts, 
    and QC counts based on the standardized metadata dataframe.
    """
    logger.info("Generating batch summary...")
    
    missing_cols = [
        c for c in [batch_col, order_col, type_col] if c not in meta.columns
    ]
    if missing_cols:
        raise ValueError(f"Missing required columns in meta: {missing_cols}")

    orders = pd.to_numeric(meta[order_col], errors="coerce")
    summary_records = []
    
    for batch_id, group in meta.groupby(batch_col):
        b_orders = orders[group.index]
        
        if b_orders.notna().any():
            order_range = f"{b_orders.min():.0f} ~ {b_orders.max():.0f}"
        else:
            order_range = "Unknown"
        
        type_counts = group[type_col].value_counts()
        
        summary_records.append({
            "Batch ID": batch_id,
            "Injection Order": order_range,
            "Actual Samples": type_counts.get("Actual Sample", 0),
            "Pooled QCs": type_counts.get("QC", 0)
        })
        
    return pd.DataFrame(summary_records)


if __name__ == "__main__":
    # Define directories (Update these paths to your specific environment)
    input_dir = os.path.abspath(os.path.join("..", "..", "data", "raw", "MetNormalizer"))
    output_dir = os.path.abspath(os.path.join("..", "..", "data", "processed", "MetNormalizer"))
    
    # Define file paths
    meta_file = os.path.join(input_dir, "sample.info.csv")
    intensity_file = os.path.join(input_dir, "data.csv")

    # 1. Execute preprocessing and sorting
    metadata_df, intensity_df = process_metnormalizer_dataset(meta_file, intensity_file)
    
    if metadata_df is not None:
        # 2. Execute summarization and print securely
        batch_summary = summarize_batch_info(meta=metadata_df)
        print("\n--- Batch Information Summary ---")
        pprint(batch_summary)
        print("---------------------------------\n")
        
        logger.info(
            f"Final Matrix Shape: {intensity_df.shape[0]} features "
            f"x {intensity_df.shape[1]} samples"
        )
        
        # 3. Export clean data
        os.makedirs(output_dir, exist_ok=True)
        meta_path = os.path.join(output_dir, "project_meta.csv")
        int_path = os.path.join(output_dir, "project_intensity.csv")

        logger.info("Saving standardized datasets...")
        metadata_df.to_csv(meta_path, index=False, encoding="utf-8-sig")
        intensity_df.to_csv(int_path, encoding="utf-8-sig", na_rep="N/A")
        
        logger.success("MetNormalizer dataset processing completed successfully.")

2026-05-25 11:04:23.752 | INFO     | __main__:process_metnormalizer_dataset:21 - Loading MetNormalizer metadata from: c:\Users\Complex\Desktop\Personal_repositories\pi-metaboqc-casestudy\data\raw\MetNormalizer\sample.info.csv
2026-05-25 11:04:23.755 | INFO     | __main__:process_metnormalizer_dataset:29 - Standardizing metadata columns...
2026-05-25 11:04:23.756 | INFO     | __main__:process_metnormalizer_dataset:46 - Formatting metadata values...
2026-05-25 11:04:23.757 | WARNING  | __main__:process_metnormalizer_dataset:50 - No 'Batch' column found. Defaulting to 'Batch-1'.
2026-05-25 11:04:23.760 | INFO     | __main__:process_metnormalizer_dataset:76 - Loading intensity file from: c:\Users\Complex\Desktop\Personal_repositories\pi-metaboqc-casestudy\data\raw\MetNormalizer\data.csv
2026-05-25 11:04:23.784 | INFO     | __main__:process_metnormalizer_dataset:100 - Sorting data by Batch and Inject Order...
2026-05-25 11:04:23.787 | INFO     | __main__:summarize_batch_info:144 - Generatin


--- Batch Information Summary ---
  Batch ID Injection Order  Actual Samples  Pooled QCs
0  Batch-1         1 ~ 177             154          23
---------------------------------

